# Cell list

The sampling keeps its computational complexity managable by dividing the box up into cells that are just a bit larger than one particle diameter, which one could achieve, for example, by setting the number of divisions to $\left\lfloor\frac{L}{\sigma}\right\rfloor$ where $L$ is simulation box size and $\sigma$ particle diameter. The reason for this is that particle pairs that are not in the same cell or neighbouring cells are definitely out of their interaction range $\sigma$, making it redundant to compute their potential energy.

For this reason we can, in fact, also safely update cells that are two apart at the same time. Cells will be two apart if we give any given cell of an update group an immediate neighbourhood of cells (as skin, if you will) of thickness one, which will be $3^d$ in count; every part of this representative cell neighbourhood can be thought of as standing for the entire parallel-updating group. The number of divisions will have to divide $3$ for this to fit cleanly, so the number of divisions we choose is $$3\left\lfloor\frac{1}{3}\frac{L}{\sigma}\right\rfloor$$

# Limits to move magnitude

The limit to the size of moves is one particle diameter, since otherwise two particles from the same cell group could potentially interact


# Illustration
![Illustration of why half a particle diameter is the worst-case upper bound for move size](cell_list_illustration.png)
<!-- <div style="text-align: center;"><img src="cell_list_illustration.png" alt="Illustration of why half a particle diameter is the worst-case upper bound for move size" width="50%"></div> -->

In [ ]:
import numpy as np

from fps import run

In [ ]:
# test,
# TODO SET UP PYTEST
from fps.indexing import __inflate_flattened_index, __flatten_multiindex
test_multi_index = np.array([51,23,10],dtype=np.int64)
assert (__inflate_flattened_index(
    __flatten_multiindex(
		test_multi_index,
		3,
		100
	),
    3,
    100
) == test_multi_index).all()

In [ ]:
from pathlib import Path

from fps.geometry import B
from xyzio import XYZ

ITERATION_COUNT = 1000
DUMP_PERIOD = 1

PARTICLE_NUMBER = 10000
SPATIAL_DIMENSION_COUNT = 3
PACKING_DENSITY = 0.6
OVERLAP_ENERGY = 5

STEP_SIZE_IN_PARTICLE_DIAMETERS = 0.25

DUMP_DIRECTORY = Path('./dumps')
DUMP_DIRECTORY.mkdir(parents=True, exist_ok=True)
FILE_PATH = DUMP_DIRECTORY / 'test_dump.xyz'

radius = (
	PACKING_DENSITY / (
	B(SPATIAL_DIMENSION_COUNT) * (PARTICLE_NUMBER/1**SPATIAL_DIMENSION_COUNT))
)**(1/SPATIAL_DIMENSION_COUNT)

x_last=None
total_acceptance_rate_history = np.array([])
for dump_period in range(int(ITERATION_COUNT/DUMP_PERIOD)):
	# doing only a few iterations at a time and dumping it for inspection
	x_last, acceptance_rates = run(
		OVERLAP_ENERGY, # overlap energy
		1, # kBT
		PACKING_DENSITY, # packing density
		PARTICLE_NUMBER, # particle number
		iteration_count=DUMP_PERIOD,
		spatial_dimension_count=SPATIAL_DIMENSION_COUNT,
		step_size_in_particle_diameters=STEP_SIZE_IN_PARTICLE_DIAMETERS,
		x_init=x_last
	)
	# remembering the acceptance rates
	total_acceptance_rate_history = np.concatenate(
		[total_acceptance_rate_history, acceptance_rates]
	)
	# dumping 
	XYZ.dump(FILE_PATH, x_last, [radius]*PARTICLE_NUMBER)

In [ ]:
import matplotlib.pyplot as plt

iteration = list(range(ITERATION_COUNT))

plt.plot(
    iteration,
    total_acceptance_rate_history,
    color='black'
)
plt.ylim(0,1)

Program Code &copy; 2026 Miriam Derla, [GNU General Public License v3.0](https://www.gnu.org/licenses/). 